# Notebook 1: Indicadores de comercio mundial

**Economia Internacional — UMET 2026**

---

## Objetivo

En la Clase 1 vimos los **hechos estilizados** del comercio internacional: como crecio, quienes participan, como se mide la apertura comercial. En este notebook vamos a trabajar con **datos reales del Banco Mundial** para calcular e interpretar algunos de esos indicadores.

## Que van a hacer

1. Cargar y explorar una base de datos de comercio internacional (10 paises, 1960-2023)
2. Ver como se calcula la **apertura comercial**, las **exportaciones e importaciones como % del PIB** y la **cuenta corriente**
3. Construir graficos para comparar paises y ver tendencias
4. **Tarea**: responder preguntas usando los datos y generando codigo con ayuda de IA

## Instrucciones

1. **Hacer una copia**: Archivo → Guardar una copia en Drive
2. **Ejecutar las celdas** en orden (Shift+Enter o boton de Play)
3. **Leer las explicaciones** entre las celdas de codigo
4. Al llegar a las celdas de **TAREA**, usar ChatGPT u otra IA para generar el codigo necesario
5. **Escribir sus interpretaciones** en las celdas de texto indicadas

> **Importante**: No necesitan saber programar. Lo que importa es que sepan **que preguntarle a la IA** y que puedan **interpretar economicamente** los resultados.

---

## Bloque 1: Conectar con el Banco Mundial y descargar datos

Vamos a traer datos directamente de la **API del Banco Mundial** (https://data.worldbank.org). Esta es la fuente oficial que usan organismos internacionales, investigadores y gobiernos. Los datos se actualizan automaticamente, asi que siempre van a tener la version mas reciente.

**No hace falta modificar nada aca, solo ejecutar las celdas.**

In [ ]:
# Librerias necesarias (ya vienen instaladas en Colab)
import pandas as pd
import matplotlib.pyplot as plt
import requests  # Para consultar APIs web

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Paleta de colores del curso
COLORES = ['#1F4E79', '#E8833A', '#27AE60', '#E74C3C', '#2E75B6',
           '#0EA5E9', '#9333EA', '#6B7280', '#D97706', '#059669']

print('Librerias cargadas correctamente')

In [ ]:
# =============================================================
# DESCARGA DE DATOS DESDE LA API DEL BANCO MUNDIAL
# Fuente: https://data.worldbank.org
# API docs: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392
# =============================================================

# Paises que vamos a analizar (codigos ISO3)
PAISES = {
    'ARG': 'Argentina', 'BRA': 'Brasil', 'CHL': 'Chile',
    'CHN': 'China', 'USA': 'Estados Unidos', 'DEU': 'Alemania',
    'KOR': 'Corea del Sur', 'MEX': 'Mexico', 'IND': 'India',
    'WLD': 'Mundo'
}

# Indicadores del Banco Mundial que vamos a descargar
INDICADORES = {
    'NE.TRD.GNFS.ZS': 'Apertura comercial (Trade % PIB)',
    'NE.EXP.GNFS.ZS': 'Exportaciones (% PIB)',
    'NE.IMP.GNFS.ZS': 'Importaciones (% PIB)',
    'BN.CAB.XOKA.GD.ZS': 'Cuenta corriente (% PIB)'
}

# Funcion para consultar la API del Banco Mundial
def descargar_indicador(paises_iso, indicador_code, indicador_nombre):
    """Descarga un indicador del Banco Mundial para una lista de paises."""
    codigos = ';'.join(paises_iso)
    url = (f'https://api.worldbank.org/v2/country/{codigos}'
           f'/indicator/{indicador_code}?format=json&per_page=5000&date=1960:2023')
    resp = requests.get(url, timeout=30)
    data = resp.json()
    
    filas = []
    if len(data) > 1 and data[1]:
        for entry in data[1]:
            if entry['value'] is not None:
                iso = entry['countryiso3code']
                filas.append({
                    'pais_iso': iso,
                    'pais': PAISES.get(iso, iso),
                    'indicador': indicador_nombre,
                    'anio': int(entry['date']),
                    'valor': round(float(entry['value']), 2)
                })
    return filas

# Descargar todos los indicadores
print('Conectando con la API del Banco Mundial...')
print(f'URL base: https://api.worldbank.org/v2/\n')

todas_las_filas = []
for cod, nombre in INDICADORES.items():
    print(f'  Descargando: {nombre}...')
    filas = descargar_indicador(list(PAISES.keys()), cod, nombre)
    todas_las_filas.extend(filas)
    print(f'    → {len(filas)} registros')

# Crear el DataFrame
df = pd.DataFrame(todas_las_filas)

print(f'\n✓ Descarga completa: {len(df)} registros')
print(f'  Paises: {", ".join(sorted(df["pais"].unique()))}')
print(f'  Periodo: {df["anio"].min()} - {df["anio"].max()}')
print(f'  Indicadores: {len(df["indicador"].unique())}')

In [ ]:
# Veamos las primeras filas para entender la estructura
df.head(10)

### Que tenemos en la base?

Acabamos de descargar datos directamente desde `api.worldbank.org`. Cada fila tiene:
- `pais`: nombre del pais
- `pais_iso`: codigo ISO3 (el que usa el Banco Mundial)
- `indicador`: que variable estamos midiendo
- `anio`: el año
- `valor`: el dato (expresado como % del PIB)

Los **4 indicadores** que descargamos son:

| Indicador | Que mide | Formula |
|-----------|----------|----------|
| Apertura comercial | Peso del comercio en la economia | (Exportaciones + Importaciones) / PIB × 100 |
| Exportaciones (% PIB) | Cuanto vende al mundo | Exportaciones / PIB × 100 |
| Importaciones (% PIB) | Cuanto compra del mundo | Importaciones / PIB × 100 |
| Cuenta corriente (% PIB) | Si el pais ahorra o se endeuda con el mundo | (Exportaciones - Importaciones + Rentas + Transferencias) / PIB × 100 |

---

## Bloque 2: Apertura comercial comparada

La **apertura comercial** mide que tan integrado esta un pais con la economia mundial. Un valor alto indica que el comercio internacional pesa mucho en su economia.

Vamos a comparar la apertura comercial de nuestros 10 paises.

In [ ]:
# Filtrar solo apertura comercial
apertura = df[df['indicador'] == 'Apertura comercial (Trade % PIB)'].copy()

# Pivotear: cada pais en una columna, cada año en una fila
apertura_pivot = apertura.pivot(index='anio', columns='pais', values='valor')

# Seleccionamos algunos paises clave para el grafico
paises_seleccion = ['Argentina', 'China', 'Estados Unidos', 'Alemania', 'Brasil', 'Mundo']
apertura_pivot[paises_seleccion].plot(
    color=COLORES[:len(paises_seleccion)],
    linewidth=2
)
plt.title('Apertura comercial (Comercio / PIB)', fontsize=14, fontweight='bold', color='#1F4E79')
plt.ylabel('% del PIB')
plt.xlabel('')
plt.legend(loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.show()

### Interpretacion

**Que nos dice este grafico?**

- **Alemania** es una economia muy abierta: su comercio supera el 80% del PIB. Es una potencia exportadora industrial.
- **China** tuvo una apertura comercial explosiva desde su ingreso a la OMC (2001), aunque bajo un poco despues de 2006 porque su PIB crecio aun mas rapido que su comercio.
- **Estados Unidos** es una economia relativamente cerrada para su tamaño (~25% del PIB). Como tiene un mercado interno enorme, el comercio pesa menos en terminos relativos.
- **Argentina** oscila entre 15% y 35%, bastante por debajo del promedio mundial. La apertura sube en los 90s (convertibilidad) y vuelve a subir post-2002 (devaluacion + boom de commodities).
- El **promedio mundial** muestra la tendencia de globalizacion: sube sostenidamente desde los 70s hasta 2008, y despues se estanca (*slowbalization*).

> **Concepto clave**: la apertura comercial depende del tamaño del pais, su estructura productiva y sus politicas comerciales. No es "buena" ni "mala" en si misma.

---

## Bloque 3: Exportaciones e importaciones

Ahora separemos exportaciones e importaciones para ver la **estructura comercial** de cada pais.

In [ ]:
# Exportaciones de Argentina, Brasil y Chile
exp = df[df['indicador'] == 'Exportaciones (% PIB)'].copy()
exp_pivot = exp.pivot(index='anio', columns='pais', values='valor')

imp = df[df['indicador'] == 'Importaciones (% PIB)'].copy()
imp_pivot = imp.pivot(index='anio', columns='pais', values='valor')

# Grafico comparativo para paises latinoamericanos
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for i, pais in enumerate(['Argentina', 'Brasil', 'Chile']):
    ax = axes[i]
    if pais in exp_pivot.columns:
        ax.plot(exp_pivot.index, exp_pivot[pais], color='#27AE60', linewidth=2, label='Exportaciones')
    if pais in imp_pivot.columns:
        ax.plot(imp_pivot.index, imp_pivot[pais], color='#E74C3C', linewidth=2, label='Importaciones')
    ax.set_title(pais, fontsize=13, fontweight='bold', color='#1F4E79')
    ax.set_ylabel('% del PIB' if i == 0 else '')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('Exportaciones e importaciones (% del PIB) — America Latina', fontsize=14, fontweight='bold', color='#1F4E79', y=1.02)
plt.tight_layout()
plt.show()

### Interpretacion

- **Chile** es mucho mas abierto que Argentina y Brasil. Su apertura comercial ronda el 60-70% del PIB. ¿Por que? Es una economia pequeña, especializada en cobre y muy integrada al comercio mundial.
- **Argentina** tuvo su pico de apertura comercial durante la convertibilidad (1991-2001) con importaciones altas, y despues de la devaluacion de 2002 las exportaciones subieron fuerte (soja, energia, manufactura).
- **Brasil** es el pais grande mas cerrado de America Latina. Su mercado interno es enorme y su apertura comercial es baja (~25%).

> **Concepto clave**: cuando las importaciones superan a las exportaciones, el pais tiene **deficit comercial**. Cuando las exportaciones superan a las importaciones, tiene **superavit**.

---

## Bloque 4: Cuenta corriente

La **cuenta corriente** es el indicador mas importante de macroeconomia abierta. Mide si un pais esta prestando al mundo (superavit) o endeudandose con el mundo (deficit).

In [ ]:
# Cuenta corriente
cc = df[df['indicador'] == 'Cuenta corriente (% PIB)'].copy()
cc_pivot = cc.pivot(index='anio', columns='pais', values='valor')

# Comparar EEUU, China, Alemania y Argentina
paises_cc = ['Estados Unidos', 'China', 'Alemania', 'Argentina']
colores_cc = ['#2E75B6', '#E74C3C', '#27AE60', '#E8833A']

fig, ax = plt.subplots(figsize=(12, 5))
for pais, color in zip(paises_cc, colores_cc):
    if pais in cc_pivot.columns:
        ax.plot(cc_pivot.index, cc_pivot[pais], color=color, linewidth=2, label=pais)

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(cc_pivot.index, 0, cc_pivot.get('Argentina', 0), alpha=0.1, color='#E8833A')
ax.set_title('Cuenta corriente (% del PIB)', fontsize=14, fontweight='bold', color='#1F4E79')
ax.set_ylabel('% del PIB')
ax.legend(loc='lower left', framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretacion

- **Estados Unidos** tiene deficit de cuenta corriente persistente desde los 80s. ¿Como lo financia? Con el privilegio del dolar como moneda de reserva mundial. El mundo le presta a EEUU comprando bonos del Tesoro.
- **China** paso de deficit a un superavit enorme (~10% del PIB en 2007), que despues se moderó. Eso refleja su estrategia exportadora y la acumulacion de reservas.
- **Alemania** tiene superavit persistente desde 2002. Exporta mucho mas de lo que importa.
- **Argentina** oscila fuertemente: deficit en los 90s (convertibilidad), superavit post-devaluacion 2002, y otra vez deficit en los 2010s.

> **Concepto clave**: un deficit de cuenta corriente no es necesariamente "malo". Depende de si financia **inversion productiva** (bueno) o **consumo** (insostenible). La identidad fundamental es: **CC = Ahorro - Inversion**.

---

## Bloque 5: Tabla resumen

Hagamos una tabla que resuma los datos mas recientes de cada pais.

In [ ]:
# Tomar el ultimo año disponible para cada pais e indicador
ultimo = df.sort_values('anio').groupby(['pais', 'indicador']).last().reset_index()

# Pivotear para tener indicadores como columnas
resumen = ultimo.pivot(index='pais', columns='indicador', values='valor')
resumen = resumen.round(1)

# Renombrar columnas para que sea mas legible
resumen.columns = ['Apertura\n(% PIB)', 'Cuenta corriente\n(% PIB)', 
                    'Exportaciones\n(% PIB)', 'Importaciones\n(% PIB)']

# Ordenar por apertura comercial
resumen = resumen.sort_values('Apertura\n(% PIB)', ascending=False)

print('Datos mas recientes por pais (ultimo año disponible):\n')
resumen

### Interpretacion

La tabla confirma lo que vimos en los graficos:
- Los paises mas abiertos (Corea, Alemania, Chile, Mexico) son economias que dependen fuertemente del comercio.
- Los mas cerrados (Brasil, Argentina, EEUU) tienen mercados internos grandes o politicas comerciales mas restrictivas.
- La cuenta corriente muestra patrones claros: superavit en exportadores netos (Alemania, China), deficit en importadores netos (EEUU).

---

## TAREA

Ahora les toca a ustedes. Usando los datos que ya estan cargados en el notebook (el DataFrame `df`), resuelvan los siguientes ejercicios.

**Como hacerlo**: copien la consigna, pegenla en ChatGPT (u otra IA) y pidanle que les genere el codigo Python. Despues peguen ese codigo en la celda y ejecutenlo. Lo importante es que:

1. Sepan **formular bien la pregunta** a la IA
2. Puedan **interpretar economicamente** el resultado

> **Tip para la IA**: diganle algo como *"Tengo un DataFrame llamado `df` con columnas: pais, indicador, anio, valor. Los indicadores son: Apertura comercial (Trade % PIB), Exportaciones (% PIB), Importaciones (% PIB), Cuenta corriente (% PIB). Necesito que..."*

---

### Tarea 1: Apertura comercial de Argentina en contexto

Construyan un grafico que muestre la **apertura comercial de Argentina** comparada con el **promedio de America Latina** (Argentina, Brasil, Chile y Mexico) y con el **promedio mundial** (Mundo).

Usen el periodo 1990-2023.

In [ ]:
# TAREA 1: Escribir el codigo aca (pedir ayuda a la IA si hace falta)
# Pista: filtrar por indicador de apertura, calcular promedio de ARG+BRA+CHL+MEX, graficar



**Interpretacion Tarea 1** (escribir aca):

1. ¿Argentina esta por encima o por debajo del promedio latinoamericano en apertura comercial?

*Respuesta:*

2. ¿En que periodos Argentina se acerco mas al promedio mundial? ¿Que politicas economicas habia en esos momentos?

*Respuesta:*

---

### Tarea 2: ¿Quien se globalizo mas rapido?

Calculen **cuanto aumento la apertura comercial** (en puntos porcentuales) entre 1990 y 2020 para cada pais. Muestren el resultado en un grafico de barras horizontal ordenado de mayor a menor.

In [ ]:
# TAREA 2: Escribir el codigo aca
# Pista: filtrar apertura comercial, tomar valores de 1990 y 2020, calcular diferencia



**Interpretacion Tarea 2** (escribir aca):

1. ¿Que pais aumento mas su apertura comercial entre 1990 y 2020? ¿Por que?

*Respuesta:*

2. ¿Hay algun pais que se haya *cerrado* comercialmente en ese periodo? ¿Como lo explican?

*Respuesta:*

---

### Tarea 3: Saldo comercial de Argentina

Calculen el **saldo comercial simplificado** de Argentina (Exportaciones - Importaciones, ambos en % del PIB) para cada año, y grafiquenlo como un grafico de barras donde los años con superavit se vean en verde y los de deficit en rojo.

In [ ]:
# TAREA 3: Escribir el codigo aca
# Pista: filtrar exportaciones e importaciones de Argentina, restar, graficar con colores condicionales



**Interpretacion Tarea 3** (escribir aca):

1. ¿En que periodos Argentina tuvo superavit comercial? ¿Y deficit? Relacionen con el contexto economico (convertibilidad, devaluacion 2002, boom de commodities, etc.)

*Respuesta:*

2. ¿Que relacion tiene el saldo comercial con el tipo de cambio real? (piensen en lo que vimos en clase sobre TCR)

*Respuesta:*

---

## Reflexion final

Respondan brevemente:

**¿Que les dicen estos datos sobre la insercion de Argentina en la economia mundial?** ¿Es un pais abierto o cerrado? ¿Su comercio internacional crece o se estanca? ¿Depende mas de las exportaciones o de las importaciones?

*Respuesta:*



---

## Entrega

1. Completar todas las celdas de TAREA
2. Responder todas las preguntas de interpretacion
3. Ejecutar todo el notebook (Menu: Entorno de ejecucion → Ejecutar todo)
4. Descargar como .ipynb (Archivo → Descargar → Descargar .ipynb)
5. Renombrar: `apellido_nombre_NB1.ipynb`
6. Enviar al docente

---

*Fuente de datos: Banco Mundial — World Development Indicators, descargados en tiempo real desde https://api.worldbank.org*

*Economia Internacional — UMET 2026*